# LLMSTU dataset enhancement — full run
Uses your locked `config.yaml`. Flow: **crop → dedup → caption → golden sample →
score → upload**. Use your **~95GB GPU** runtime.

For the real 104k-frame pass, prefer the shard-by-shard loop in `docs/GPU_95GB.md`
(resumable, session-safe). This notebook runs it in one go — fine for a slice or a
`LIMIT`ed sanity pass; everything is resumable if it stops.

## 1. Setup

In [ ]:
# PRIVATE repo? add GITHUB_TOKEN to Colab Secrets (key icon). Public clones w/o one.
REPO = "vaelkokach/LLMSTU-pipeline"
import os
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return None
if REPO and not os.path.isdir("/content/LLMSTU-pipeline"):
    gh=_secret("GITHUB_TOKEN"); auth=f"{gh}@" if gh else ""
    !git clone https://{auth}github.com/{REPO}.git /content/LLMSTU-pipeline
if os.path.isdir("/content/LLMSTU-pipeline"):
    %cd /content/LLMSTU-pipeline
print("cwd:", os.getcwd())

In [ ]:
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate "bitsandbytes>=0.43.0" \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow numpy
# For max throughput on the full run also: !pip install "vllm>=0.11.0" flash-attn --no-build-isolation
import torch; print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Set HF_TOKEN in Colab Secrets"
from huggingface_hub import login; login(os.environ["HF_TOKEN"])

## 2. Config\nRespects your locked `config.yaml` (model, crop, prompt, schema, dedup). Keep `LIMIT` small for a sanity pass, set `None` for the full run.

In [ ]:
from pathlib import Path
from llmstu.config import load
cfg = load("config.yaml")
LIMIT = 500   # crops to caption; set None for the FULL run
print("model:", cfg.caption.model_id, "| prompt:", cfg.caption.prompt_name,
      "| schema:", cfg.caption.schema_name)
print("crop_mode:", cfg.crop.crop_mode, "| dedup thr:", cfg.dedup.hamming_threshold,
      "| backend:", cfg.caption.backend)

## 3. Detect + crop students\nDownloads frames from HF (first run only), then crops. For the full run this pulls all 21 shards (~35GB).

In [ ]:
from llmstu import dataset_io as io, crop as crop_mod
frames_dir = io.download_frames(cfg.data.source_repo, cfg.data.source_subdir,
                                Path(cfg.data.workdir)/"frames_dl", os.environ["HF_TOKEN"])
crop_mod.run(io.iter_frames(frames_dir, cfg.data.frames_glob),
             Path("work/crops"), cfg.crop, manifest_path=Path("work/crops_manifest.jsonl"))

In [ ]:
# peek at a few crops
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob("work/crops/**/*.jpg", recursive=True))[:6]:
    display(IPImage(p, width=200))

## 4. Remove near-duplicate crops (1-fps footage)\nCollapses runs of near-identical same-seat crops BEFORE captioning — big cost saver. Tune `dedup.hamming_threshold` in `config.yaml`.

In [ ]:
from llmstu import dedup
stats = dedup.dedup_manifest(Path("work/crops_manifest.jsonl"), Path("work/crops"),
                             Path("work/crops_manifest_dedup.jsonl"),
                             hamming_threshold=cfg.dedup.hamming_threshold,
                             cell_px=cfg.dedup.cell_px, hash_size=cfg.dedup.hash_size)
print(stats)  # kept / dropped / reduction

## 5. Pseudo-label the DEDUPED crops with Qwen3.5\nCaptions `work/crops_manifest_dedup.jsonl`. For the full run set `LIMIT=None` and consider `cfg.caption.backend='vllm'`.

In [ ]:
from llmstu import caption as caption_mod
caption_mod.run(Path("work/crops_manifest_dedup.jsonl"), Path("work/crops"),
                Path("work/pseudo_labels.jsonl"), cfg.caption, limit=LIMIT)

In [ ]:
import json, pandas as pd
rows=[json.loads(l) for l in open("work/pseudo_labels.jsonl")]
df=pd.DataFrame(rows); print(len(df),"labeled")
df[["crop_path","activity","gaze_direction","engagement_level","model_confidence"]].head(10)

## 6. Build the golden-eval sample (stratified + hard subset)\nSelf-contained payload. Download, hand-label in `index.html`, then come back for step 7.

In [ ]:
!python scripts/03_sample_eval.py --labels work/pseudo_labels.jsonl --crops work/crops \
    --n 1500 --stratify-fields engagement_level,activity,gaze_direction \
    --min-per-class 40 --hard-frac 0.2 --embed
from google.colab import files
!cd labeling && zip -qr /content/labeling.zip index.html data.json
files.download("/content/labeling.zip")

## 7. Score pseudo-labels vs golden\nAfter hand-labeling, upload your `golden.json` (cell below), then score. Reports representative vs hard accuracy.

In [ ]:
from google.colab import files
up = files.upload()   # choose your exported golden.json
import shutil; shutil.move(list(up)[0], "labeling/golden.json"); print("saved labeling/golden.json")

In [ ]:
!python scripts/07_eval_golden.py --golden labeling/golden.json
from IPython.display import HTML
HTML(open("work/eval/golden_report.html").read())

## 8. Package + push\nUploads sharded crops + `metadata.jsonl` (caption per image) + parquet + golden set to your crops repo.

In [ ]:
!python scripts/04_upload.py --crops work/crops --labels work/pseudo_labels.jsonl \
    --golden labeling/golden.json --repo CHANGE_ME/your-crops-dataset